### What is masked attention?

**Attention is all you need paper:** self-attention layers in the decoder allow each position in the decoder to attend to
all positions in the decoder up to and including that position. We need to prevent leftward
information flow in the decoder to preserve the auto-regressive property. We implement this
inside of scaled dot-product attention by masking out (setting to −∞) all values in the input
of the softmax which correspond to illegal connections.

aka Causal Attention



<div>
<img src="../helpful_imgs/masked_attn.png" width="600">
</div>

In [2]:
import torch.nn as nn
import torch
torch.manual_seed(123)

class SelfAttention_v2(nn.Module):
    def __init__(self,d_in,d_out,qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
    
    def forward(self,X):
        keys = self.W_key(X)
        queries = self.W_query(X)
        values = self.W_value(X)
        attn_scores = queries @ keys.T
        d_k = keys.shape[-1]
        attn_weights = torch.softmax(input=(attn_scores/(d_k**0.5)),dim=-1)
        context_vec = attn_weights @ values
        return context_vec



In [3]:
# “Your journey starts with one step”
inputs = torch.tensor(
[[0.43, 0.15, 0.89], # Your (x^1)
[0.55, 0.87, 0.66], # journey (x^2)
[0.57, 0.85, 0.64], # starts (x^3)
[0.22, 0.58, 0.33], # with (x^4)
[0.77, 0.25, 0.10], # one (x^5)
[0.05, 0.80, 0.55]] # step (x^6)
)
d_in,d_out = inputs.shape[1],2
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)


In [ ]:
queries = sa_v2.W_query(inputs) # W_query weights
keys = sa_v2.W_key(inputs) # W_key weights
attn_scores = queries @ keys.T # unnormalized attention scores
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1) # normalized attention scores
print(attn_weights)

tensor([[0.1717, 0.1762, 0.1761, 0.1555, 0.1627, 0.1579],
        [0.1636, 0.1749, 0.1746, 0.1612, 0.1605, 0.1652],
        [0.1637, 0.1749, 0.1746, 0.1611, 0.1606, 0.1651],
        [0.1636, 0.1704, 0.1702, 0.1652, 0.1632, 0.1674],
        [0.1667, 0.1722, 0.1721, 0.1618, 0.1633, 0.1639],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<SoftmaxBackward0>)


##### If you refer the above diagram, it resembles Lower Traingular matrix from Linear Algebra 
<img src="https://i.ytimg.com/vi/8C9PRvJrxTI/maxresdefault.jpg" width=300>

In [5]:

# TODO: Use Pytorch tril function to create a mask where the values above the diagonal are zero

context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [ ]:

# Now just multiply this mask with attention weights to zero out values above the diagonal

masked_simple = attn_weights * mask_simple
print(masked_simple)

# Note we used * here because we want element wise matrix multiplication aka Hadamard Product
# We generated a simple mask

tensor([[0.1717, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1636, 0.1749, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1637, 0.1749, 0.1746, 0.0000, 0.0000, 0.0000],
        [0.1636, 0.1704, 0.1702, 0.1652, 0.0000, 0.0000],
        [0.1667, 0.1722, 0.1721, 0.1618, 0.1633, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<MulBackward0>)


Now what we can do is normalize it again (renormalize) by: dividing each element in each row by the sum in each row

Which is what softmax does

$\text{Softmax}(x_{i}) = \frac{\exp(x_i)}{\sum_j \exp(x_j)}$

In [12]:
row_sums = masked_simple.sum(-1,True)
renorm_masked_simple = masked_simple / row_sums
renorm_masked_simple

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4833, 0.5167, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3190, 0.3408, 0.3402, 0.0000, 0.0000, 0.0000],
        [0.2445, 0.2545, 0.2542, 0.2468, 0.0000, 0.0000],
        [0.1994, 0.2060, 0.2058, 0.1935, 0.1953, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<DivBackward0>)

In short what we did was

1. Calc. Attention scores
2. Calc. Attention weights by normalizing Attn scores using softmax
3. Created a mask
4. Calc. Masked Attention score by : Multiplying the mask with attention weights 
5. Calc Masked attention Weights by (*renormalizing it*)


In [ ]:

# TODO: However we implement causal attn using less steps
#*  According to the paper: "We implement this inside of scaled dot-product attention 
#* by masking out (setting to −∞) all values in the input
#* of the softmax which correspond to illegal connections.

queries = sa_v2.W_query(inputs) # W_query weights
keys = sa_v2.W_key(inputs) # W_key weights
attn_scores = queries @ keys.T # unnormalized attention scores


# TODO: Generate mask s.t. you set the enteries above the main diagnoal -inf instead of 0
# Make a *upper triangular matrix where diagonal is excluded i.e. (Diagonal !=1) 
mask = torch.triu(torch.ones(context_length, context_length),diagonal=1)
print(f"Upper Triangular matrix:\n {mask} \n\n")
final_masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
# Use masked_fill to replace 1's to -inf
print(f"Final Mask: \n{final_masked}\n\n")

attn_weights = torch.softmax(final_masked / keys.shape[-1]**0.5, dim=1) # normalized attention scores
print(attn_weights)

Upper Triangular matrix:
 tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]]) 


Final Mask: 
tensor([[0.3111,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.1655, 0.2602,   -inf,   -inf,   -inf,   -inf],
        [0.1667, 0.2602, 0.2577,   -inf,   -inf,   -inf],
        [0.0510, 0.1080, 0.1064, 0.0643,   -inf,   -inf],
        [0.1415, 0.1875, 0.1863, 0.0987, 0.1121,   -inf],
        [0.0476, 0.1192, 0.1171, 0.0731, 0.0477, 0.0966]],
       grad_fn=<MaskedFillBackward0>)


tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4833, 0.5167, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3190, 0.3408, 0.3402, 0.0000, 0.0000, 0.0000],
        [0.2445, 0.2545, 0.2542, 0.2468, 0.0000, 0.0000],
        [0.1994, 0.2060, 0.2058, 0.1935, 0.1953, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=

We only did 1 added step here compared to scaled dot product attention
1. Calc Attention Score
2. Generate Mask
3. Normalize using Softmax 

Why did this work?
-> Softmax converts inputs to probability distribution. When it sees a negative value in the row it automatically treats it as zero because $e^{-\inf}$ approaches 0.

In [49]:
values = sa_v2.W_value(inputs)
context_vec = attn_weights @ values
context_vec

tensor([[-0.4519,  0.2216],
        [-0.5874,  0.0058],
        [-0.6300, -0.0632],
        [-0.5675, -0.0843],
        [-0.5526, -0.0981],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)

### Implementing causal attention class

Where it can handle batches consisting of more than 1 input and support dropout also.


In [56]:
batched_inputs = torch.stack((inputs,inputs),dim=0)
print(batched_inputs.shape)


torch.Size([2, 6, 3])


In [67]:
class CausalAttention(nn.Module):
    def __init__(self,d_in,d_out,context_length,dropout,qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_Q = nn.Linear(d_in,d_out,qkv_bias)
        self.W_K = nn.Linear(d_in,d_out,qkv_bias)
        self.W_V = nn.Linear(d_in,d_out,qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask',torch.triu(torch.ones(context_length, context_length),diagonal=1))

    
    def forward(self,x):
        b,num_tokens,d_in = x.shape
        self.num_tokens = num_tokens
        keys = self.W_K(x)
        queries = self.W_Q(x)
        values = self.W_V(x)

        attn_score = queries @ keys.transpose(1,2)
        attn_score.masked_fill_(self.mask.bool()[:num_tokens,:num_tokens],-torch.inf)
        d_k = keys.shape[-1]
        attn_weights = torch.softmax(attn_score / d_k**0.5,dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec




In [69]:
torch.manual_seed(123)
context_length = batched_inputs.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batched_inputs)
print("context_vecs.shape:", context_vecs.shape)

context_vecs.shape: torch.Size([2, 6, 2])


In [70]:
context_vecs

tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)

In [71]:
ca.num_tokens

6

In PyTorch, operations with a trailing underscore are performed in-place, avoiding unnecessary memory copies.

Eg. masked_fill_